# Conclusion & Executive Summary

## 1. Key Findings & Feature Engineering

**Approach to Missing Data:** Several clinical features in this dataset are heavily incomplete — `slope` (~34% missing), `ca` (~66%), and `thal` (~53%). Rather than impute these large gaps (which would fabricate signal and bias the model), we adopted a model that handles missing values **natively**: `HistGradientBoostingClassifier`. Each gap is left in place and the algorithm learns, at every decision split, the most informative direction to route a missing value.

**Fuller Use of the Data:** An earlier iteration trained on only a subset of features and excluded several strong clinical predictors. This version restores them — most importantly `cp` (chest pain type), `oldpeak` (exercise-induced ST depression), `exang` (exercise-induced angina), `slope`, and `restecg`. Categorical features are handled natively via pandas `category` dtype, removing the need for manual one-hot encoding.

## 2. Data-Quality Audit (Plain English)

Using model-explainability tools (SHAP), we audited *why* the model was making its predictions — and found it was partly cheating.

**The problem.** The data combines patient records from four different hospitals. One hospital (in Switzerland) recorded a patient's cholesterol as **"0"** whenever the value was actually missing — but a cholesterol of zero is medically impossible. That same hospital happened to have an extremely high rate of heart disease (93% of its patients). So the model quietly learned a shortcut: *"if cholesterol = 0, predict heart disease."* It wasn't learning medicine — it was secretly detecting **which hospital a patient came from**. The hospital label itself offered the same shortcut.

**Why that matters.** A model that takes shortcuts can look accurate while being untrustworthy. This one was partly memorizing "which hospital" instead of understanding "which symptoms signal heart disease." Faced with a patient from a new clinic, that shortcut would be useless — or misleading.

**How we caught it.** SHAP revealed features behaving backwards — for example, *lower* cholesterol appearing to predict *more* disease. That clinical nonsense was the tell that pointed us to the underlying data artifact.

**The fix.** We told the model that an impossible "0" means "unknown" (not a real low value), and we removed the hospital label entirely so the model must rely on genuine clinical symptoms. The result: essentially the same accuracy, **higher recall**, and predictions now grounded in real medical signals rather than a data quirk.

## 3. Model Performance & Medical Optimization

**The Model:** A gradient-boosted tree ensemble (`HistGradientBoostingClassifier`) with `class_weight='balanced'` to counter class imbalance. After the data-quality fixes it achieves **83.2% accuracy** and a **0.903 ROC-AUC** at the default threshold — virtually unchanged from before the fixes, confirming the model's true discriminating power never depended on the artifact.

**The Threshold Solution:** To prioritize patient safety and minimize dangerous false negatives, the decision threshold was tuned from 0.50 down to **0.35** — flagging a patient as at-risk on even moderate suspicion. This deliberately trades a small amount of precision for higher recall, the correct trade-off for a clinical screening tool.

## 4. Final Metrics

| Metric | Value | Interpretation |
| --- | --- | --- |
| Overall Accuracy | **83.2%** | Honest model, no site leakage |
| ROC-AUC | **0.903** | Strong separation of disease vs. no-disease |
| Class 1 Recall (@0.35) | **0.93** | Catches 93% of all actual heart-disease cases |
| Class 1 Precision (@0.35) | **0.81** | A reliable positive predictive value |

**Top Predictive Drivers** (by SHAP): now dominated by genuine clinical features such as `age`, `chol`, `oldpeak`, and `thal` — rather than the hospital-site artifact removed during the audit.

## 5. Clinical Impact & Next Steps

By accepting a manageable rate of false alarms, the model maximizes its primary objective: catching high-risk cardiac patients before they leave the clinic. Critically, it now does so using legitimate clinical signal rather than a data-collection quirk, so it stands a far better chance of generalizing to a new hospital.

**Completed:**
- **SHAP explainability** — global (beeswarm), local (waterfall), and dependence plots for per-patient, model-agnostic explanations of *why* an individual was flagged.
- **Streamlit demo app** — an interactive web form that loads the trained model, predicts risk at the 0.35 threshold, and shows a per-patient SHAP explanation.

**Next steps:**
- **Threshold optimization** driven explicitly by clinical cost (cost of a missed case vs. a false alarm) to formally justify the operating point.
- **Public deployment** of the Streamlit app (e.g. Streamlit Community Cloud) so it can be shared via a link.